# 03 — Stage 2 U-Net training

**One job:** mine detector-specific negatives and optionally train one crop model

Run from top to bottom. Slow actions are off by default.

In [ ]:
%pip install -q ultralytics pycocotools segmentation-models-pytorch

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = Path("/content/drive/MyDrive/Google Colab/Solar Filament Segmentation Challenge")
sys.path.insert(0, str(PROJECT_DIR))
%load_ext autoreload
%autoreload 2

from src.config import DEFAULT_PIPELINE_SETTINGS, TRAINING_SETTINGS
from src.data import copy_raw_data_to_local, load_records, make_split, verify_or_write_split

LOCAL_DATA_DIR = copy_raw_data_to_local(PROJECT_DIR / "data/raw")
TRAIN_IMAGES_DIR = LOCAL_DATA_DIR / "train/train_images"
TEST_IMAGES_DIR = LOCAL_DATA_DIR / "test/test_images"
ANNOTATIONS_PATH = LOCAL_DATA_DIR / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
SPLIT_MANIFEST_PATH = PROJECT_DIR / "data/processed/split_manifest.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
MINE_HARD_NEGATIVES = False
TRAIN = False
MODEL_NAME = "b5"  # "crop_unet" or "b5"
DETECTOR_NAME = "yolo26x"
YOLO_CHECKPOINT = PROJECT_DIR / "models/stage1/yolo26x_filament_detection/weights/best.pt"
HARD_NEGATIVE_CACHE = PROJECT_DIR / "data/processed/hard_negatives/hard_negatives_yolo26x_conf005.json"
MASK_THRESHOLD = 0.45  # what inference uses; checkpoint selection now scores at the deployed threshold

## 1. Hard-negative mining

Caches are detector-specific. Mining uses confidence 0.05; the optional score floor is applied only when building samples.

In [ ]:
from src.data import mine_hard_negatives

coco, records = load_records(ANNOTATIONS_PATH, TRAIN_IMAGES_DIR)
_, train_records, validation_records = verify_or_write_split(records, SPLIT_MANIFEST_PATH)
train_positive_samples = [{"physical_id": r["physical_id"], "file_name": r["file_name"], "image_path": r["image_path"],
                           "annotation": annotation, "proposal_box": None} for r in train_records for annotation in r["annotations"]]
if HARD_NEGATIVE_CACHE.exists():
    hard_negatives = json.loads(HARD_NEGATIVE_CACHE.read_text())
elif MINE_HARD_NEGATIVES:
    from ultralytics import YOLO

    settings = {**DEFAULT_PIPELINE_SETTINGS, **TRAINING_SETTINGS, "yolo_mining_confidence": 0.05}
    hard_negatives = mine_hard_negatives(train_records, YOLO(str(YOLO_CHECKPOINT)), settings, device,
                                         round(len(train_positive_samples) * settings["hard_negative_ratio"]))
    HARD_NEGATIVE_CACHE.write_text(json.dumps(hard_negatives))
else:
    hard_negatives = []
    print("No cache found; mining skipped.")
print(f"Positive crops: {len(train_positive_samples)}, cached hard negatives: {len(hard_negatives)}")

## 2. Datasets, loaders, and crop sanity check

In [ ]:
from torch.utils.data import DataLoader
from src.data import FilamentCropDataset
from src.viz import show_crop

settings = {**DEFAULT_PIPELINE_SETTINGS, **TRAINING_SETTINGS}
negative_samples = [{"physical_id": x["physical_id"], "file_name": x["file_name"], "image_path": TRAIN_IMAGES_DIR / x["file_name"],
                     "annotation": None, "proposal_box": x["proposal_box"]} for x in hard_negatives if x["score"] >= settings["negative_mining_floor"]]
train_dataset = FilamentCropDataset(train_positive_samples + negative_samples, settings, augment=True)
validation_samples = [{"physical_id": r["physical_id"], "file_name": r["file_name"], "image_path": r["image_path"],
                       "annotation": annotation, "proposal_box": None} for r in validation_records for annotation in r["annotations"]]
validation_dataset = FilamentCropDataset(validation_samples, settings, augment=False)
inputs, target, _ = validation_dataset[0]
show_crop(inputs, target)

## 3. Model recipes

| Recipe | Selection metric | Saved result |
|---|---|---|
| Plain CropUNet, GroupNorm, from scratch | Positive-crop oracle Dice | 0.8183 at epoch 16 |
| EfficientNet-B5 U-Net, ImageNet, scse | Positive-crop oracle Dice | 0.8283 at epoch 13 |
| Proposal-fine-tuned CropUNet + 8-way TTA | Official full-pipeline PQ | 0.4381 |

Since 2026-09-19 training selects on **official PQ**, not oracle Dice: the Dice validation crops are all positives with oracle prompts, so it cannot see the false positives that dominate the score. `center_jitter` also widened from 0.10 to 0.20 to match measured proposal offsets (median 0.030, p90 0.229).

## 4. Training

The validator runs the whole pipeline each epoch. Detector boxes are cached once, so only Stage 2 is recomputed.

In [ ]:
from src.inference import cache_proposals, official_pq_for_model
from src.models import build_model, train_crop_unet

if TRAIN:
    from ultralytics import YOLO

    run_dir = PROJECT_DIR / "models/stage2" / datetime.now().strftime(f"%Y%m%d_%H%M%S_{MODEL_NAME}")
    run_dir.mkdir(parents=True, exist_ok=False)
    validation_paths = sorted({r["physical_id"]: r["image_path"] for r in validation_records}.values())
    proposals = cache_proposals(YOLO(str(YOLO_CHECKPOINT)), validation_paths, device, DEFAULT_PIPELINE_SETTINGS)
    pq_validator = lambda unet: official_pq_for_model(unet, validation_records, proposals, device, MASK_THRESHOLD, DEFAULT_PIPELINE_SETTINGS)
    train_loader = DataLoader(train_dataset, batch_size=settings["batch_size"], shuffle=True, num_workers=settings["num_workers"], pin_memory=True)
    validation_loader = DataLoader(validation_dataset, batch_size=settings["batch_size"], shuffle=False, num_workers=settings["num_workers"], pin_memory=True)
    model = build_model(MODEL_NAME).to(device)
    history = train_crop_unet(model, train_loader, validation_loader, device, settings, run_dir / "best.pt", run_dir / "history.csv", pq_validator)
else:
    print("Training skipped.")

## 5. Oracle evaluation

> Warning: oracle crop Dice and IoU do not predict the official end-to-end score. Use notebook 04 for model decisions.

In [ ]:
oracle_results = pd.DataFrame([
    {"checkpoint": "CropUNet", "oracle_dice": 0.8183, "oracle_iou": 0.6986},
    {"checkpoint": "EfficientNet-B5 U-Net", "oracle_dice": 0.8283, "oracle_iou": 0.7122},
])
display(oracle_results)